# Module 2: SAML SSO Validation (Optional)

## Overview

This notebook validates SAML SSO configuration for your LangSmith deployment. Use this if your IdP only supports SAML or if enterprise policy requires SAML.

**⚠️ SAFETY NOTICE:** This notebook is **READ-ONLY**. It performs validation checks only and does NOT modify any infrastructure, Helm values, secrets, or deployments. All operations are safe to run against production environments.

**Prerequisites:**
- Module 1 deployment is healthy and accessible
- DNS configured and resolving correctly
- TLS certificate valid and trusted
- Ingress configured and working
- IdP team has provided SAML metadata or metadata URL

## What We'll Validate

1. ✅ Environment configuration (SAML settings, redacted)
2. ✅ Preflight checks (tools, kubectl, namespace, Helm release)
3. ✅ Current auth configuration (without leaking secrets)
4. ✅ Ingress/TLS preconditions (domain, HTTPS)
5. ✅ SAML metadata validation (URL reachability, XML parsing, required attributes)
6. ✅ Deployment verification (pods, logs, endpoints)
7. ✅ Common failure signatures
8. ✅ Support bundle pointers

**Estimated time:** 30-45 minutes

**Important:** 
- This notebook never prints secrets. All sensitive values are redacted.
- This notebook does NOT modify any resources. It is safe for production use.


In [ ]:
# Bootstrap environment
import sys
from pathlib import Path

# Add notebooks directory to path so we can import shared as a package
possible_paths = [
    Path.cwd().parent,  # If cwd is module-2, go up one level to notebooks
    Path.cwd(),  # If cwd is already notebooks
    Path.cwd() / "notebooks",  # If cwd is workspace root
]

notebooks_path = None
for path in possible_paths:
    if path and (path / "shared" / "_bootstrap.py").exists():
        notebooks_path = path
        break

if not notebooks_path:
    notebooks_path = Path.cwd() / "notebooks"
    if not (notebooks_path / "shared" / "_bootstrap.py").exists():
        raise RuntimeError(f"Could not find notebooks/shared directory. Current dir: {Path.cwd()}")

# Add notebooks directory to path so 'shared' can be imported as a package
if str(notebooks_path) not in sys.path:
    sys.path.insert(0, str(notebooks_path))

from shared._bootstrap import bootstrap

# Run bootstrap
bootstrap_info = bootstrap()
artifacts_dir = Path(bootstrap_info['artifacts_dir'])
print(f"\nArtifacts directory: {artifacts_dir}")


## 1. Configuration

Load and validate SAML configuration from environment variables. All secrets are redacted in output.


In [ ]:
import os
import json
from pathlib import Path
from shared._validation import require_env, print_config, redact, ok, warn
from shared._shell import run

# Required SAML configuration variables
required_vars = [
    "NAMESPACE",
    "SAML_METADATA_URL",  # OR SAML_METADATA_FILE (one must be provided)
    "LANGSMITH_DOMAIN",
]

# Optional but recommended
optional_vars = [
    "SAML_ENTITY_ID",
    "SAML_EMAIL_ATTRIBUTE",
    "SAML_NAME_ATTRIBUTE",
    "SAML_GROUPS_ATTRIBUTE",
]

print("### Loading SAML Configuration\n")

# Load required variables
config = {}
missing = []

for var in required_vars:
    value = os.environ.get(var, "").strip()
    if not value:
        missing.append(var)
    config[var] = value

# Check if SAML_METADATA_FILE is provided as alternative
saml_metadata_file = os.environ.get("SAML_METADATA_FILE", "").strip()
if not config.get("SAML_METADATA_URL") and not saml_metadata_file:
    missing.append("SAML_METADATA_URL or SAML_METADATA_FILE")

if missing:
    raise RuntimeError(f"❌ Missing required environment variables: {', '.join(missing)}\n"
                      f"💡 Copy env-samples/oidc.env.example to your .env file and fill in SAML values")

# Load optional variables
for var in optional_vars:
    config[var] = os.environ.get(var, "").strip()

# Set defaults for optional variables
if not config.get("SAML_EMAIL_ATTRIBUTE"):
    config["SAML_EMAIL_ATTRIBUTE"] = "email"
if not config.get("SAML_NAME_ATTRIBUTE"):
    config["SAML_NAME_ATTRIBUTE"] = "name"
if not config.get("SAML_GROUPS_ATTRIBUTE"):
    config["SAML_GROUPS_ATTRIBUTE"] = "groups"

# Print configuration (redacted)
print_config(config, redact_keys=set())

ok("Configuration loaded")

# Validate metadata source
if config.get("SAML_METADATA_URL"):
    metadata_url = config["SAML_METADATA_URL"]
    if not metadata_url.startswith("https://"):
        warn("SAML metadata URL should use HTTPS")
    print(f"\n💡 Using metadata URL: {metadata_url}")
elif saml_metadata_file:
    metadata_path = Path(saml_metadata_file)
    if not metadata_path.exists():
        raise RuntimeError(f"❌ SAML metadata file not found: {saml_metadata_file}")
    print(f"\n💡 Using metadata file: {saml_metadata_file}")
else:
    raise RuntimeError("❌ Either SAML_METADATA_URL or SAML_METADATA_FILE must be provided")

domain = config["LANGSMITH_DOMAIN"]
print(f"\n💡 Verify these values match your IdP configuration:")
print(f"   - Entity ID: {config.get('SAML_ENTITY_ID', 'N/A')}")
print(f"   - Metadata URL/File: {config.get('SAML_METADATA_URL', saml_metadata_file)}")
print(f"   - Domain: {domain}")


## Safety Check: Verify Environment

Before proceeding with validation, confirm you're working with the correct environment and that auth configuration is appropriate to validate.


In [ ]:
# Safety check: Verify environment and auth configuration state
from shared._cloud_helpers import get_cloud_provider, get_region, get_identity
from shared._validation import ok, warn
from shared._shell import run
import json

provider = get_cloud_provider()
region = get_region()
identity = get_identity()

print("### Environment Safety Check\n")

# Show current environment
provider_display = provider.upper()
print(f"Cloud Provider: {provider_display}")
print(f"Region: {region}")

if provider == "aws":
    print(f"Account ID: {identity.get('Account', 'N/A')}")
    print(f"User ARN: {identity.get('Arn', 'N/A')}")
elif provider == "azure":
    print(f"Subscription ID: {identity.get('SubscriptionId', identity.get('Account', 'N/A'))}")
    print(f"Subscription Name: {identity.get('SubscriptionName', 'N/A')}")

print("\n" + "=" * 60)
print("⚠️  IMPORTANT: This notebook is READ-ONLY")
print("=" * 60)
print("\nThis notebook will:")
print("  ✅ Validate SAML configuration")
print("  ✅ Check deployment status")
print("  ✅ Inspect current auth settings (secrets redacted)")
print("  ✅ Collect support bundles")
print("\nThis notebook will NOT:")
print("  ❌ Modify Helm values or releases")
print("  ❌ Create or update secrets")
print("  ❌ Restart pods or deployments")
print("  ❌ Change any infrastructure")
print("\n" + "=" * 60)

# Check if auth is already configured
print("\n### Checking Current Auth Configuration State\n")
namespace = config.get("NAMESPACE", "")
helm_release = os.environ.get("HELM_RELEASE", "langsmith")

# Check for auth-related secrets
result = run(
    ["kubectl", "get", "secrets", "-n", namespace, "-o", "json"],
    check=False,
    stream=False
)

auth_configured = False
if result.returncode == 0:
    secrets = json.loads(result.stdout)
    auth_secrets = [s for s in secrets.get("items", [])
                   if any(keyword in s.get("metadata", {}).get("name", "").lower()
                         for keyword in ["auth", "saml", "sso"])]
    
    if auth_secrets:
        auth_configured = True
        ok(f"Found {len(auth_secrets)} auth-related secret(s) - auth appears configured")
        print("   💡 This validation will check if your SAML configuration matches existing setup")
    else:
        warn("No auth-related secrets found - auth may not be configured yet")
        print("   💡 This validation will verify your SAML configuration is ready to apply")

# Check Helm values for auth config
result = run(
    ["helm", "get", "values", helm_release, "-n", namespace, "--output", "json"],
    check=False,
    stream=False
)

if result.returncode == 0:
    try:
        values = json.loads(result.stdout)
        if "auth" in str(values).lower() or "saml" in str(values).lower():
            if not auth_configured:
                auth_configured = True
            ok("Helm values contain auth configuration")
        else:
            warn("No auth configuration found in Helm values")
    except json.JSONDecodeError:
        pass

if auth_configured:
    print("\n" + "=" * 60)
    print("⚠️  Auth is already configured in this environment")
    print("=" * 60)
    print("\nThis validation will:")
    print("  - Verify your SAML settings match the existing configuration")
    print("  - Check if authentication is working correctly")
    print("  - Identify any configuration mismatches")
    print("\n💡 If you need to CHANGE auth configuration, use Helm upgrade separately")
    print("   This notebook only validates, it does not modify configuration")
else:
    print("\n" + "=" * 60)
    print("ℹ️  Auth not yet configured")
    print("=" * 60)
    print("\nThis validation will:")
    print("  - Verify your SAML settings are correct")
    print("  - Check prerequisites (DNS, TLS, ingress)")
    print("  - Validate IdP metadata")
    print("\n💡 After validation passes, apply configuration using Helm upgrade")
    print("   This notebook only validates, it does not apply configuration")

ok("Environment safety check complete")
print("\n✅ Safe to proceed with validation")


## 2. Preflight Checks

Same as OIDC notebook - verify tools, kubectl context, namespace, and Helm release.


In [ ]:
from shared._validation import ok, warn
from shared._k8s_helpers import require_namespace, namespace_exists
from shared._shell import run
from shared._cloud_helpers import get_cloud_provider, get_region

provider = get_cloud_provider()
region = get_region()
namespace = config["NAMESPACE"]

print("### Preflight Checks\n")

# Check kubectl is available
print("1. Checking kubectl...")
result = run(["kubectl", "version", "--client", "--short"], check=False, stream=False)
if result.returncode == 0:
    ok("kubectl is available")
    print(f"   {result.stdout.strip()}")
else:
    raise RuntimeError("❌ kubectl is not available or not working")

# Check kubectl context
print("\n2. Checking kubectl context...")
result = run(["kubectl", "config", "current-context"], check=False, stream=False)
if result.returncode == 0:
    context = result.stdout.strip()
    ok(f"Current context: {context}")
else:
    warn("Could not determine kubectl context")

# Check namespace exists
print(f"\n3. Checking namespace '{namespace}'...")
if namespace_exists(namespace):
    ok(f"Namespace '{namespace}' exists")
else:
    raise RuntimeError(f"❌ Namespace '{namespace}' does not exist. Complete Module 1 first.")

# Check Helm release
print(f"\n4. Checking Helm release...")
helm_release = os.environ.get("HELM_RELEASE", "langsmith")
result = run(
    ["helm", "list", "-n", namespace, "--output", "json"],
    check=False,
    stream=False
)

if result.returncode == 0:
    try:
        releases = json.loads(result.stdout)
        release_names = [r.get("name") for r in releases]
        if helm_release in release_names:
            ok(f"Helm release '{helm_release}' exists")
        else:
            raise RuntimeError(f"❌ Helm release '{helm_release}' not found")
    except json.JSONDecodeError:
        warn("Could not parse Helm release list")

ok("Preflight checks complete")


## 3. Inspect Current Auth Configuration

Examine the current authentication configuration without leaking secrets.


In [ ]:
print("### Inspecting Current Auth Configuration\n")

# Check for auth-related environment variables in deployments
print("1. Checking deployment environment variables...")
result = run(
    ["kubectl", "get", "deployments", "-n", namespace, "-o", "json"],
    check=False,
    stream=False
)

if result.returncode == 0:
    deployments = json.loads(result.stdout)
    auth_vars_found = False
    
    for deployment in deployments.get("items", []):
        name = deployment.get("metadata", {}).get("name", "")
        containers = deployment.get("spec", {}).get("template", {}).get("spec", {}).get("containers", [])
        
        for container in containers:
            env_vars = container.get("env", [])
            auth_env = [e for e in env_vars if any(keyword in e.get("name", "").upper() for keyword in ["AUTH", "SAML", "SSO"])]
            
            if auth_env:
                auth_vars_found = True
                print(f"\n   Deployment: {name}")
                for env in auth_env:
                    env_name = env.get("name", "")
                    if "SECRET" in env_name.upper() or "PASSWORD" in env_name.upper():
                        print(f"     - {env_name}: <redacted>")
                    elif env.get("valueFrom"):
                        print(f"     - {env_name}: <from secret/configmap>")
    
    if not auth_vars_found:
        warn("No auth-related environment variables found")

# Check for auth-related secrets (names only)
print("\n2. Checking for auth-related secrets...")
result = run(
    ["kubectl", "get", "secrets", "-n", namespace, "-o", "json"],
    check=False,
    stream=False
)

if result.returncode == 0:
    secrets = json.loads(result.stdout)
    auth_secrets = [s for s in secrets.get("items", [])
                   if any(keyword in s.get("metadata", {}).get("name", "").lower()
                         for keyword in ["auth", "saml", "sso"])]
    
    if auth_secrets:
        ok(f"Found {len(auth_secrets)} auth-related secret(s)")
        for secret in auth_secrets:
            name = secret.get("metadata", {}).get("name", "")
            print(f"   - {name} (values not displayed)")

ok("Auth configuration inspection complete (no secrets displayed)")


## 4. Validate Ingress/TLS Preconditions

Verify domain resolution, HTTPS accessibility, and TLS certificate validity.


In [ ]:
import socket
import ssl
import requests

domain = config["LANGSMITH_DOMAIN"]
print(f"### Validating Ingress/TLS for {domain}\n")

# 1. DNS Resolution
print("1. Checking DNS resolution...")
try:
    ip_address = socket.gethostbyname(domain)
    ok(f"Domain resolves to: {ip_address}")
except socket.gaierror as e:
    raise RuntimeError(f"❌ DNS resolution failed for {domain}: {e}")

# 2. HTTPS Reachability
print(f"\n2. Checking HTTPS reachability...")
https_url = f"https://{domain}"

try:
    response = requests.get(https_url, timeout=10, verify=True, allow_redirects=True)
    ok(f"HTTPS accessible: {response.status_code}")
except requests.exceptions.SSLError as e:
    warn(f"SSL verification failed: {e}")
    print("   💡 Certificate may be self-signed or invalid")
except requests.exceptions.RequestException as e:
    raise RuntimeError(f"❌ Could not connect to {domain}: {e}")

# 3. TLS Certificate Check
print(f"\n3. Checking TLS certificate...")
try:
    context = ssl.create_default_context()
    with socket.create_connection((domain, 443), timeout=10) as sock:
        with context.wrap_socket(sock, server_hostname=domain) as ssock:
            cert = ssock.getpeercert()
            subject = dict(x[0] for x in cert['subject'])
            
            import datetime
            not_after = datetime.datetime.strptime(cert['notAfter'], '%b %d %H:%M:%S %Y %Z')
            days_until_expiry = (not_after - datetime.datetime.now()).days
            
            if days_until_expiry > 30:
                ok(f"Certificate valid for {days_until_expiry} more days")
            elif days_until_expiry > 0:
                warn(f"Certificate expires in {days_until_expiry} days")
            else:
                raise RuntimeError(f"❌ Certificate expired")
except Exception as e:
    warn(f"Could not verify TLS certificate: {e}")

ok("Ingress/TLS preconditions validated")


## 5. SAML Metadata Validation

Validate SAML metadata URL reachability, XML parsing, and required attributes.


In [ ]:
import xml.etree.ElementTree as ET
import requests

print("### Validating SAML Metadata\n")

metadata_url = config.get("SAML_METADATA_URL", "")
metadata_file = os.environ.get("SAML_METADATA_FILE", "").strip()

# 1. Fetch or Load Metadata
print("1. Loading SAML metadata...")
metadata_xml = None

if metadata_url:
    print(f"   Fetching from URL: {metadata_url}")
    try:
        response = requests.get(metadata_url, timeout=10, verify=True)
        if response.status_code == 200:
            ok("Metadata URL accessible")
            metadata_xml = response.text
        else:
            raise RuntimeError(f"❌ Metadata URL returned {response.status_code}")
    except requests.exceptions.RequestException as e:
        raise RuntimeError(f"❌ Could not fetch metadata URL: {e}")
elif metadata_file:
    print(f"   Loading from file: {metadata_file}")
    try:
        with open(metadata_file, "r") as f:
            metadata_xml = f.read()
        ok("Metadata file loaded")
    except Exception as e:
        raise RuntimeError(f"❌ Could not load metadata file: {e}")

if not metadata_xml:
    raise RuntimeError("❌ No metadata XML available")

# 2. Parse XML
print("\n2. Parsing SAML metadata XML...")
try:
    # Register namespaces
    namespaces = {
        'md': 'urn:oasis:names:tc:SAML:2.0:metadata',
        'ds': 'http://www.w3.org/2000/09/xmldsig#',
    }
    
    root = ET.fromstring(metadata_xml)
    ok("Metadata XML is valid")
except ET.ParseError as e:
    raise RuntimeError(f"❌ Invalid XML: {e}")

# 3. Extract Entity Descriptor
print("\n3. Extracting entity information...")
entity_id = None
try:
    entity_descriptor = root.find('.//md:EntityDescriptor', namespaces)
    if entity_descriptor is not None:
        entity_id = entity_descriptor.get('entityID')
        if entity_id:
            ok(f"Entity ID found: {entity_id}")
            if config.get("SAML_ENTITY_ID") and entity_id != config.get("SAML_ENTITY_ID"):
                warn(f"Entity ID mismatch: config={config.get('SAML_ENTITY_ID')}, metadata={entity_id}")
        else:
            warn("Entity ID not found in metadata")
    else:
        warn("EntityDescriptor not found in metadata")
except Exception as e:
    warn(f"Could not extract entity information: {e}")

# 4. Extract IDP SSO Descriptor
print("\n4. Extracting IdP SSO descriptor...")
try:
    idp_sso = root.find('.//md:IDPSSODescriptor', namespaces)
    if idp_sso is not None:
        ok("IdP SSO descriptor found")
        
        # Extract SSO endpoints
        sso_endpoints = idp_sso.findall('.//md:SingleSignOnService', namespaces)
        if sso_endpoints:
            print(f"   Found {len(sso_endpoints)} SSO endpoint(s):")
            for endpoint in sso_endpoints:
                location = endpoint.get('Location', '')
                binding = endpoint.get('Binding', '')
                print(f"     - {binding}: {location}")
        else:
            warn("No SSO endpoints found")
    else:
        warn("IDPSSODescriptor not found - may not be IdP metadata")
except Exception as e:
    warn(f"Could not extract IdP SSO descriptor: {e}")

# 5. Extract Certificates
print("\n5. Checking for signing certificates...")
try:
    certificates = root.findall('.//ds:X509Certificate', namespaces)
    if certificates:
        ok(f"Found {len(certificates)} certificate(s)")
        for i, cert in enumerate(certificates):
            cert_text = cert.text.strip() if cert.text else ""
            if cert_text:
                print(f"   Certificate {i+1}: {len(cert_text)} characters")
            else:
                warn(f"Certificate {i+1} is empty")
    else:
        warn("No signing certificates found")
        print("   💡 IdP must provide signing certificate for assertion validation")
except Exception as e:
    warn(f"Could not extract certificates: {e}")

# 6. Validate Required Attributes
print("\n6. Validating attribute configuration...")
print(f"   Expected email attribute: {config['SAML_EMAIL_ATTRIBUTE']}")
print(f"   Expected name attribute: {config['SAML_NAME_ATTRIBUTE']}")
print(f"   Expected groups attribute: {config['SAML_GROUPS_ATTRIBUTE']}")

ok("SAML metadata validation complete")
print("\n💡 Verify your IdP sends these attributes in SAML assertions:")
print(f"   - {config['SAML_EMAIL_ATTRIBUTE']} (required)")
print(f"   - {config['SAML_NAME_ATTRIBUTE']} (optional)")
print(f"   - {config['SAML_GROUPS_ATTRIBUTE']} (optional, for role mapping)")


In [ ]:
from shared._shell import run

print("### Checking for Common SAML Failure Signatures\n")

# Get pod names
result = run(
    ["kubectl", "get", "pods", "-n", namespace, "-o", "jsonpath={.items[*].metadata.name}"],
    check=False,
    stream=False
)

if result.returncode == 0 and result.stdout.strip():
    pod_names = result.stdout.strip().split()
    api_pods = [p for p in pod_names if any(keyword in p.lower() for keyword in ["api", "server", "backend"])]
    
    if not api_pods:
        api_pods = pod_names[:2]
    
    failure_patterns = {
        "Missing Attributes": [
            "missing attribute",
            "attribute not found",
            "email attribute",
            "required attribute",
        ],
        "Signature Validation": [
            "signature validation failed",
            "invalid signature",
            "certificate",
            "signing key",
        ],
        "Assertion Expired": [
            "assertion expired",
            "notonorafter",
            "clock skew",
            "timeout",
        ],
        "Entity ID Mismatch": [
            "entity id",
            "issuer mismatch",
            "audience",
        ],
        "Metadata Issues": [
            "metadata",
            "xml parse",
            "invalid metadata",
        ],
    }
    
    found_issues = []
    
    for pod_name in api_pods[:2]:
        try:
            log_result = run(
                ["kubectl", "logs", pod_name, "-n", namespace, "--tail=200"],
                check=False,
                stream=False
            )
            
            if log_result.returncode == 0:
                logs_lower = log_result.stdout.lower()
                
                for category, patterns in failure_patterns.items():
                    for pattern in patterns:
                        if pattern in logs_lower:
                            # Check if it's actually an error (not just a log message)
                            lines = log_result.stdout.split("\n")
                            error_lines = [line for line in lines 
                                         if pattern in line.lower() 
                                         and any(err in line.lower() for err in ["error", "fail", "invalid", "missing"])]
                            
                            if error_lines and category not in found_issues:
                                found_issues.append(category)
                                warn(f"Potential {category} issue found in {pod_name} logs")
                                print(f"   Pattern: '{pattern}'")
                                # Don't print full log line as it may contain sensitive data
                                break
        except Exception:
            pass
    
    if not found_issues:
        ok("No common SAML failure signatures found in logs")
    else:
        print(f"\n💡 Found potential issues: {', '.join(found_issues)}")
        print("   Review logs manually for details:")
        print(f"   kubectl logs <pod-name> -n {namespace} --tail=100 | grep -i saml")
else:
    warn("Could not retrieve pod names")

print("\n💡 Common SAML failure causes:")
print("   1. Missing required attributes in assertion")
print("   2. Certificate mismatch or expired certificate")
print("   3. Clock skew between LangSmith and IdP")
print("   4. Entity ID mismatch")
print("   5. Attribute name mismatch")
print("\n   See docs/shared/auth_troubleshooting.md for detailed troubleshooting")


## 4. Deployment Verification & Support Bundle

Same as OIDC notebook - verify pods, check logs, collect support bundle.


In [ ]:
from shared._k8s_helpers import get_pods, wait_for_deployments_ready
from datetime import datetime
import requests

print("### Deployment Verification\n")

# 1. Pod Readiness
print("1. Checking pod readiness...")
require_namespace(namespace)

try:
    wait_for_deployments_ready(namespace, timeout="5m")
    ok("All deployments ready")
except Exception as e:
    warn(f"Some deployments may not be ready: {e}")

pods_output = get_pods(namespace)
print("\nPod Status:")
print(pods_output)

# 2. Test Endpoint Auth Behavior
print(f"\n2. Testing endpoint auth behavior...")
domain = config["LANGSMITH_DOMAIN"]
test_url = f"https://{domain}/api/v1/me"

try:
    response = requests.get(test_url, timeout=10, verify=True, allow_redirects=False)
    if response.status_code in [401, 403]:
        ok(f"Endpoint requires authentication ({response.status_code})")
    elif response.status_code in [301, 302, 307, 308]:
        redirect_location = response.headers.get("Location", "")
        if "login" in redirect_location.lower() or "saml" in redirect_location.lower():
            ok("Endpoint redirects to authentication")
        else:
            warn(f"Endpoint redirects but not to auth: {redirect_location}")
    else:
        warn(f"Unexpected status code: {response.status_code}")
except requests.exceptions.RequestException as e:
    warn(f"Could not test endpoint: {e}")

# 3. Support Bundle
print(f"\n3. Collecting support bundle...")
timestamp = datetime.now().strftime("%Y%m%d-%H%M%S")
support_dir = artifacts_dir / f"saml-support-{timestamp}"
support_dir.mkdir(exist_ok=True)

result = run(
    ["kubectl", "get", "pods", "-n", namespace, "-o", "jsonpath={.items[*].metadata.name}"],
    check=False,
    stream=False
)

if result.returncode == 0 and result.stdout.strip():
    pod_names = result.stdout.strip().split()
    api_pods = [p for p in pod_names if any(keyword in p.lower() for keyword in ["api", "server", "backend"])]
    
    for pod_name in (api_pods[:3] if api_pods else pod_names[:3]):
        try:
            log_result = run(
                ["kubectl", "logs", pod_name, "-n", namespace, "--tail=200"],
                check=False,
                stream=False
            )
            if log_result.returncode == 0:
                log_file = support_dir / f"{pod_name}-logs.txt"
                with open(log_file, "w") as f:
                    f.write(log_result.stdout)
                print(f"   ✅ Saved logs for {pod_name}")
        except Exception:
            pass

ok(f"Support bundle saved to: {support_dir}")
print("\n💡 Include pod logs and configuration when contacting support")
print("   See docs/shared/auth_troubleshooting.md for complete bundle procedure")
